# Log File Anomaly Detection System

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest

# Sample log data (emulating SSH log format)
sample_logs = [
    "Jan 11 12:30:12 server sshd[1234]: Failed password for invalid user admin from 192.168.1.10 port 22",
    "Jan 11 12:30:14 server sshd[1234]: Accepted password for user root from 192.168.1.10 port 22",
    "Jan 11 01:14:20 server sshd[1234]: Failed password for invalid user guest from 192.168.1.23 port 22",
    "Jan 11 01:14:45 server sshd[1234]: Failed password for invalid user test from 192.168.1.23 port 22",
    "Jan 11 01:15:03 server sshd[1234]: Failed password for invalid user admin from 192.168.1.23 port 22",
    "Jan 11 01:15:22 server sshd[1234]: Accepted password for user alice from 192.168.1.23 port 22",
    "Jan 11 05:22:12 server sshd[1234]: Failed password for root from 10.0.0.5 port 22",
    "Jan 11 05:22:25 server sshd[1234]: Failed password for root from 10.0.0.5 port 22",
    "Jan 11 05:22:30 server sshd[1234]: Failed password for root from 10.0.0.5 port 22",
    "Jan 11 08:10:33 server sshd[1234]: Accepted password for user bob from 192.168.1.5 port 22",
]

# Parsing function
def parse_logs(logs):
    data = []
    for line in logs:
        parts = line.split()
        time_str = "2024 " + " ".join(parts[0:3])
        time = datetime.strptime(time_str, "%Y %b %d %H:%M:%S")
        status = "Failed" if "Failed" in line else "Accepted"
        ip = next((part for part in parts if part.count('.') == 3), None)
        data.append({"timestamp": time, "status": status, "ip": ip})
    return pd.DataFrame(data)

df = parse_logs(sample_logs)

# Feature engineering
df['hour'] = df['timestamp'].dt.hour
df['fail'] = df['status'].apply(lambda x: 1 if x == "Failed" else 0)
df['ip_int'] = df['ip'].apply(lambda ip: int(ip.split('.')[-1]))  # simple IP encoding

# Apply Isolation Forest
features = df[['hour', 'fail', 'ip_int']]
model = IsolationForest(contamination=0.25, random_state=42)
df['anomaly'] = model.fit_predict(features)

# Plotting
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='hour', y='ip_int', hue='anomaly', style='status', palette='coolwarm')
plt.title("Anomaly Detection on SSH Logs")
plt.xlabel("Hour of Day")
plt.ylabel("Encoded IP (last octet)")
plt.grid(True)
plt.tight_layout()
plt.show()

df[['timestamp', 'ip', 'status', 'anomaly']]
##Legend:

##anomaly = -1 → flagged as suspicious

##anomaly = 1 → normal behavior

